# Midterm Analysis

**1. What challenges did you encounter when provisioning and executing machine learning workflows in a managed cloud environment (such as Colab Enterprise or Compute Engine) compared to local or standard free-tier notebooks?**

Setting up Colab Enterprise took more steps than free Colab, because I had to redeem the education credit, select the right project, enable the Vertex AI API, and create a runtime. Packages like ucimlrepo had to be installed with %pip install in each new runtime, since installs don't persist. My Task 1 code also showed that Colab Enterprise is a managed container rather than a plain VM: the project ID printed correctly, but instance name, zone, and machine type returned "Not Found." I also hit a NameError after cells ran out of order, which shows how the runtime keeps state and can restart. Unlike a local notebook, idle runtimes keep using credits, so I had to remember to stop the runtime. In return, I got dedicated compute (4 CPUs) and proof of execution tied to my GCP project.

**2. How did you identify and handle implicit missing values (such as -200 sentinel indicators) across the sensor columns, and how did your imputation strategy affect model stability?**

I counted the -200 sentinels per column. NMHC(GT) was missing 90.23% (8,443 rows), CO(GT) 17.99%, NO2(GT) 17.55%, NOx(GT) 17.52%, and every other column 3.91% (366 rows). I replaced -200 with NaN, confirmed zero sentinels remained, and dropped NMHC(GT) because imputing 90% of a column would mostly invent data. For the other features I used SimpleImputer with the median, because the pollutants are right-skewed (skewness 1.4 to 1.7 for CO, C6H6 and NOx). Imputation shrank the spread of the most-imputed columns: CO(GT) std fell from 1.453 to 1.323, NOx(GT) from 212.98 to 195.09, and NO2(GT) from 48.37 to 43.95. For the targets I dropped the rows with missing CO(GT) (9,357 → 7,674 rows) instead of imputing labels. The models were stable: softmax train/test accuracy was 0.887/0.872, and regression train/test R² was 0.911/0.913, so there was no sign of overfitting from the imputation.

**3. Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing and training steps in your cloud pipeline.**

The ColumnTransformer sends each column type through its own steps: median imputation and StandardScaler for the 12 numeric columns, and most-frequent imputation and one-hot encoding for hour, day of week and month. That produced 55 columns in Task 5 (12 numeric + 24 + 7 + 12 encoded). The Pipeline chains those steps with the model, so one fit and one predict run the whole workflow. This prevents leakage because I split first, and the pipeline then learned medians, means, standard deviations and categories from the training rows only (6,139 rows). The 1,535 test rows were only transformed with those saved values. Rebuilding the same build_preprocessor for the classification (54 features) and regression (51 features, with the target removed and drop="first") tasks also kept the notebook easy to rerun in the cloud.

**4. How did comparing StandardScaler versus MinMaxScaler impact your model training convergence, gradient descent optimization, and final evaluation metrics?**
I compared both scalers on softmax regression and two SGD models, using the same split. StandardScaler gave better results for every model. Softmax went from 0.8723 accuracy and 0.8731 macro F1 (StandardScaler) to 0.8515 and 0.8527 (MinMaxScaler). SGD with log_loss and eta0=0.01 dropped the most, from 0.8619 accuracy and 0.8627 macro F1 to 0.8235 and 0.8223. SGD with the optimal learning rate fell from 0.8502 to 0.8410. MinMaxScaler also changed the iteration counts: softmax needed 67 iterations instead of 77, and SGD (eta0=0.01) stopped after 29 epochs instead of 46. Stopping sooner was not better convergence, because SGD stopped earlier at a worse solution, probably because the early-stopping tolerance was met while the weights were still poor. I think StandardScaler works better here because the pollutant columns are right-skewed with outliers, and MinMax scaling squeezes most values into a narrow band near 0, which makes those features less informative under a fixed learning rate and regularization strength. Fit times were small and noisy (all under 2.5 seconds), so I would not draw conclusions from them. I kept StandardScaler for the final pipeline.

**5. How does configuring SGDClassifier differ from standard closed-form solvers, and what advantages do probability outputs and evaluation metrics provide when assessing classification performance?**

The lbfgs solver in softmax regression optimizes over the whole dataset and needed 77 iterations. SGDClassifier takes many cheap, noisy steps, so its result depends on the loss, learning-rate schedule, eta0, alpha and penalty. In my results, SGD with log_loss and constant eta0=0.01 was the best SGD model at 0.8619 accuracy and 0.8627 macro F1 in 46 epochs. That was close to, but below, softmax at 0.8723 accuracy and 0.8731 macro F1. A very small learning rate (eta0=0.0001) climbed slowly and was still rising after 30 epochs, ending at only 0.8235 accuracy. log_loss and modified_huber give probabilities, which let you see model confidence and set decision thresholds. hinge gives labels only. Macro F1 and the confusion matrix show more than accuracy: Low and High were rarely confused (3 and 2 errors for softmax), but Medium had the lowest F1 (0.82) and absorbed most errors, for example 67 Low rows and 52 High rows predicted as Medium.

**6. Based on your regularized regression models (Ridge versus Lasso), how did adjusting the regularization hyperparameter affect feature selection and coefficient shrinkage, and what operational environmental insights do these findings reveal?**

Ridge shrank coefficients smoothly but never to zero: at alpha=10 all 51 coefficients were still non-zero, and the sum of absolute coefficients only fell from 10.045 to 8.838 with test R² essentially unchanged (0.9123). Lasso removed features as alpha rose, leaving 43 non-zero at 0.001, 14 at 0.01, 5 at 0.05, 4 at 0.1, 1 at 1.0 and 0 at 10. Test R² fell with it, from 0.9119 to 0.8934, 0.8736, 0.8651, 0.3222 and finally -0.002. Linear Regression, Ridge and cross-validated Lasso all scored about 0.9125 test R², so regularization added stability and interpretability rather than accuracy. The cross-validated Lasso kept 50 of 51 features and dropped only hour_1. At alpha=0.2 the survivors were C6H6(GT) (0.732), NOx(GT) (0.342), PT08.S1(CO) (0.160) and NO2(GT) (0.005). In the OLS model, correlated sensors carried opposite-sign weights, PT08.S4(NO2) at +0.83 and PT08.S2(NMHC) at −0.63, so I wouldn't over-read which sensor survives. Operationally, CO tracks the other reference pollutants most strongly, and the largest time effects were higher CO at hours 19 and 20 and in December, and lower CO at hours 3, 6 and 7. That suggests evening and winter monitoring matters most, though these are associations, not causes.